# Tunable TWPA calibration

This notebook only orchestrates the workflow. Acquisition, checkpointing, same-flux normalization, scoring, and plotting live in `QickworkspaceV2.experiments.twpa.workflow`.

Workflow: pump-off reference → coarse pump/flux scan → offline ranking → optional fine scan. Hardware scans checkpoint each completed flux row and also save in `finally`, while guaranteeing pump-off on normal completion, interruption, or error.

## 1. Imports

In [14]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np

from QickworkspaceV2.config.system_cfg import config_list
from QickworkspaceV2.core.base_experiment import BaseExperiment
from QickworkspaceV2.experiments.twpa import (
    TWPACalibrator,
    TWPASweepPlan,
    analyze_twpa_run,
    latest_twpa_run_directory,
    new_twpa_run_directory,
    plot_twpa_summary,
    rank_twpa_candidates,
)
from QickworkspaceV2.instruments import BaseInstrumentManager
from QickworkspaceV2.tools.system_tool import ExperimentConfig

## 2. One source of truth

All scan parameters are here. Pump power is the room-temperature source setting; record the calibrated line attenuation separately before interpreting device-plane power.

In [15]:
DATA_PATH = Path(r"D:\Labber_Data\Jay\test\twpa")
RUN_ROOT = DATA_PATH / "calibration_runs"
RESUME_RUN_DIR = None  # Example: RUN_ROOT / "20260805_131547"
QUBIT = "Q4"

QICK_HOST = "192.168.10.82"
QICK_PORT = 8888
QICK_PROXY = "myqick"
YOKO_ADDRESS = "USB0::0x0B21::0x0039::91S522309::INSTR"
YOKO_NAME = "twpa_flux"
PUMP_ADDRESS = "192.168.10.43"
PUMP_NAME = "twpa_pump"

plan = TWPASweepPlan(
    signal_start_mhz=6700,
    signal_stop_mhz=7000,
    signal_steps=301,
    flux_values=np.linspace(0.57e-3, 0.65e-3, 21),
    pump_freqs_hz=np.linspace(10.7e9, 11.0e9, 16),
    pump_powers_dbm=np.array([16.0, 18.0, 20.0]),
    resonator_gain=0.1,
    py_avg=1,
    pump_settle_s=0.1,
    yoko_mode="current",
    target_f_min_hz=6.70e9,
    target_f_max_hz=7.00e9,
    gain_threshold_db=12.0,
    gain_target_db=15.0,
    ripple_limit_db=5.0,
)

base_config = ExperimentConfig(config_list).get_qubit(QUBIT)
run_cfg = plan.build_run_cfg(base_config)
run_dir = Path(RESUME_RUN_DIR) if RESUME_RUN_DIR else new_twpa_run_directory(RUN_ROOT)
run_dir.mkdir(parents=True, exist_ok=True)
reference_path = run_dir / "reference.nc"
scan_path = run_dir / "scan.nc"
print(run_dir)

D:\Labber_Data\Jay\test\twpa\calibration_runs\20260806_154511


## 3. Connect hardware

This is the only setup cell that creates hardware state. Acquisition cells depend explicitly on `calibrator`, `reference_path`, and `scan_path`.

In [16]:
soc, soccfg = BaseExperiment.connect_pyro4(
    ns_host=QICK_HOST,
    ns_port=QICK_PORT,
    proxy_name=QICK_PROXY,
    data_path=str(DATA_PATH),
)

instruments = BaseInstrumentManager()
yoko = instruments.add_yoko(
    YOKO_NAME,
    YOKO_ADDRESS,
    auto_limits=False,
    limits={"current": (-1e-3, 1e-3), "voltage": (-2.0, 2.0)},
    current_ramp_step=1e-6,
    voltage_ramp_step=1e-5,
    ramp_interval=0.001,
)
pump = instruments.add_sgs100a(PUMP_NAME, PUMP_ADDRESS)
calibrator = TWPACalibrator(
    run_cfg,
    plan,
    pump_source=pump,
    instrument_manager=instruments,
    yoko_name=YOKO_NAME,
)
print(instruments.status)

QICK library version mismatch: 0.2.381 remote (the board), 0.2.405 local (the PC)
                        This may cause errors, usually KeyError in QickConfig initialization.
                        If this happens, you must bring your versions in sync.


Pyro.NameServer PYRO:Pyro.NameServer@0.0.0.0:8888
myqick PYRO:obj_a0fa10b880d442359a82595a954ab5e1@192.168.10.82:44291
[BaseExperiment] Session activated: QICK@192.168.10.82:8888/myqick, data_path='D:\\Labber_Data\\Jay\\test\\twpa'
Connected to: YOKOGAWA,GS210,91S522309,2.02
Connected to: Rohde&Schwarz,SGS100A,1416.0505k02/114212,4.2.76.0-4.30.046.295
yoko: twpa_flux address: USB0::0x0B21::0x0039::91S522309::INSTR | output: off | value: voltage=0.0 V, ramp=I_step=1e-06, V_step=1e-05, interval=0.001
sgs100a: twpa_pump address: TCPIP::192.168.10.43::INSTR | output: on | value: power=-20.0 dBm, frequency=11000000000.0 Hz


In [17]:
yoko.current = 0.65e-3

Yoko current 0 A -> 650 uA:   0%|          | 0/651 [00:00<?, ?step/s]

In [31]:
pump.frequency=11.3e9
pump.power = 18
pump.on()

## 4. Pump-off reference

A reference trace is measured at every flux value. Rerun this cell after interruption to resume missing rows. Use `overwrite=True` only when intentionally replacing the reference.

In [ ]:
reference_path = calibrator.acquire_reference(reference_path)
print("reference:", reference_path)

## 5. Coarse scan

The order minimizes flux ramps: pump power → flux → pump frequency. The pump is turned off in `finally`, including KeyboardInterrupt. Rerunning resumes incomplete points.

In [ ]:
scan_path = calibrator.acquire_scan(
    scan_path,
    reference_path=reference_path,
)
print("scan:", scan_path)

## 6. Offline analysis — independent of hardware

This cell can run after a kernel restart. Set `analysis_run_dir` explicitly for an older run; by default it loads the newest directory containing both files. Gain is normalized with the pump-off trace at the same flux.

In [ ]:
from pathlib import Path
import json
from QickworkspaceV2.experiments.twpa import (
    analyze_twpa_run, latest_twpa_run_directory,
    plot_twpa_summary, rank_twpa_candidates,
)

analysis_root = Path(r"D:\Labber_Data\Jay\test\twpa") / "calibration_runs"
analysis_run_dir = latest_twpa_run_directory(analysis_root)
analysis = analyze_twpa_run(
    analysis_run_dir / "scan.nc",
    analysis_run_dir / "reference.nc",
    target_f_min_hz=6.70e9,
    target_f_max_hz=7.00e9,
    gain_threshold_db=12.0,
    gain_target_db=15.0,
    ripple_limit_db=5.0,
)
candidates = rank_twpa_candidates(analysis, count=5)
(analysis_run_dir / "candidates.json").write_text(
    json.dumps(candidates, indent=2), encoding="utf-8"
)
for index, point in enumerate(candidates, 1):
    print(
        f"#{index}: pump={point['pump_freq']/1e9:.6f} GHz, "
        f"power={point['pump_power']:+.1f} dBm, flux={point['ifbl']/1e-6:.1f} uA, "
        f"median={point['median_gain_db']:.2f} dB, "
        f"p10={point['p10_gain_db']:.2f} dB, ripple={point['ripple_db']:.2f} dB, "
        f"coverage={point['coverage_fraction']:.1%}"
    )
fig, axes = plot_twpa_summary(analysis, candidates[0])
plt.show()

## 7. Optional fine scan

After reviewing the coarse result, create a new run centered on the selected candidate. A new pump-off reference is required because the fine scan uses a new flux grid. Set `RUN_FINE_SCAN = True` deliberately.

In [ ]:
RUN_FINE_SCAN = False

if RUN_FINE_SCAN:
    selected = candidates[0]
    fine_plan = plan.refined(selected)
    fine_cfg = fine_plan.build_run_cfg(base_config)
    fine_dir = new_twpa_run_directory(RUN_ROOT)
    fine_calibrator = TWPACalibrator(
        fine_cfg,
        fine_plan,
        pump_source=pump,
        instrument_manager=instruments,
        yoko_name=YOKO_NAME,
    )
    fine_reference = fine_calibrator.acquire_reference(fine_dir / "reference.nc")
    fine_scan = fine_calibrator.acquire_scan(
        fine_dir / "scan.nc", reference_path=fine_reference
    )
    print(fine_dir)

## 8. Emergency / final shutdown

Normal scans already turn the pump off. This cell is intentionally idempotent; choose a safe flux value only if your device procedure calls for ramping the bias after calibration.

In [ ]:
calibrator.shutdown(flux_safe_value=None)
print("Pump OFF")